# 18. Activation and Loss Backward | 激活与损失反向（Task0 解答版）

材料来源：[datawhalechina/llm-algo-leetcode · 18_Activation_and_Loss_Backward.ipynb](https://github.com/datawhalechina/llm-algo-leetcode/blob/main/02_PyTorch_Algorithms/18_Activation_and_Loss_Backward.ipynb)

In [1]:
import torch
import torch.nn.functional as F
print('torch', torch.__version__)

torch 2.9.1+cu128


In [2]:
def relu_backward(grad_out, x):
    """手写 ReLU 的反向传播。

    Args:
        grad_out: 上游梯度，与 x 形状相同。
        x: ReLU 前的输入张量。x == 0 时本题按梯度 0 处理。

    Returns:
        传回输入 x 的梯度。
    """
    if grad_out.shape != x.shape:
        raise ValueError('grad_out 和 x 必须具有相同形状')
    if grad_out.device != x.device:
        raise ValueError('grad_out 和 x 必须位于同一 device')
    # TODO 1: ReLU 反向门控掩码（x == 0 按约定置 0）
    mask = (x > 0).to(grad_out.dtype)
    return grad_out * mask


def softmax_ce_loss_and_grad(logits, labels, reduction="mean"):
    """计算 mean / sum reduction 的交叉熵及其 logits 梯度。"""
    if logits.ndim != 2 or labels.ndim != 1 or logits.size(0) != labels.size(0):
        raise ValueError('logits 应为 [batch, classes]，labels 应为 [batch]')
    if logits.device != labels.device:
        raise ValueError('logits 和 labels 必须位于同一 device')
    if reduction not in ("mean", "sum"):
        raise ValueError('reduction 只能是 mean 或 sum')
    if not logits.is_floating_point():
        raise TypeError('logits 必须是浮点张量')
    if labels.dtype not in (torch.int32, torch.int64):
        raise TypeError('labels 必须是整数类别索引')
    if labels.numel() and (labels.min() < 0 or labels.max() >= logits.size(1)):
        raise ValueError('labels 超出类别范围')
    # TODO 2: log_softmax 保证数值稳定，再按 labels 构造 one_hot
    log_probs = F.log_softmax(logits, dim=-1)
    probs = log_probs.exp()
    one_hot = torch.zeros_like(probs)
    one_hot.scatter_(1, labels.unsqueeze(1), 1.0)
    # TODO 3: 逐样本 loss → 按 reduction 聚合 → logits 梯度 = probs - one_hot
    per_sample_loss = -(one_hot * log_probs).sum(dim=1)
    loss = per_sample_loss.mean() if reduction == "mean" else per_sample_loss.sum()
    grad = probs - one_hot
    if reduction == "mean":
        grad = grad / logits.size(0)
    return loss, grad

In [3]:
def test_activation_and_loss_backward():
    """验证两个局部 backward 的数值、边界行为和输入契约。"""
    x = torch.tensor([-2.0, -0.5, 0.0, 1.0, 3.0], requires_grad=True)
    upstream = torch.tensor([0.5, -1.0, 2.0, 0.25, -0.75])
    F.relu(x).backward(upstream)
    manual_relu = relu_backward(upstream, x.detach())
    assert torch.allclose(x.grad, manual_relu), "ReLU backward 不一致"
    assert torch.isfinite(manual_relu).all(), "ReLU 梯度包含 NaN 或 Inf"

    logits = torch.tensor([[1.0, 0.5, -0.2], [0.2, -0.3, 1.2]], requires_grad=True)
    labels = torch.tensor([0, 2])
    loss, manual_grad = softmax_ce_loss_and_grad(logits, labels)
    ce = F.cross_entropy(logits, labels)
    ce.backward()
    assert torch.allclose(loss, ce.detach(), atol=1e-6), "CrossEntropy loss 不一致"
    assert torch.allclose(logits.grad, manual_grad, atol=1e-6), "CrossEntropy backward 不一致"
    assert torch.isfinite(loss) and torch.isfinite(manual_grad).all(), "CrossEntropy 结果包含 NaN 或 Inf"

    sum_logits = logits.detach().clone().requires_grad_()
    sum_loss = F.cross_entropy(sum_logits, labels, reduction='sum')
    sum_loss.backward()
    assert torch.allclose(sum_loss, loss.detach() * labels.numel(), atol=1e-6)
    assert torch.allclose(sum_logits.grad, manual_grad * labels.numel(), atol=1e-6)
    manual_sum_loss, manual_sum_grad = softmax_ce_loss_and_grad(logits.detach(), labels, reduction='sum')
    assert torch.allclose(manual_sum_loss, sum_loss.detach(), atol=1e-6)
    assert torch.allclose(manual_sum_grad, sum_logits.grad, atol=1e-6)

    edge_logits = torch.tensor([[1000.0, 0.0, -1000.0]], requires_grad=True)
    edge_labels = torch.tensor([0])
    edge_loss, edge_grad = softmax_ce_loss_and_grad(edge_logits, edge_labels)
    edge_ref = F.cross_entropy(edge_logits, edge_labels)
    edge_ref.backward()
    assert torch.isfinite(edge_loss) and torch.isfinite(edge_grad).all(), "大数值输入产生了非有限值"
    assert torch.allclose(edge_loss, edge_ref.detach(), atol=1e-6)
    assert torch.allclose(edge_logits.grad, edge_grad, atol=1e-6)

    try:
        softmax_ce_loss_and_grad(torch.zeros(1, 3), torch.tensor([3]))
    except ValueError as error:
        assert "类别范围" in str(error)
    else:
        raise AssertionError("越界标签应该触发 ValueError")

    try:
        softmax_ce_loss_and_grad(torch.zeros(1, 3), torch.tensor([0]), reduction='median')
    except ValueError as error:
        assert "reduction" in str(error)
    else:
        raise AssertionError("非法 reduction 应该触发 ValueError")

    try:
        relu_backward(torch.ones(2), torch.ones(3))
    except ValueError as error:
        assert "相同形状" in str(error)
    else:
        raise AssertionError("ReLU 输入形状不一致应该触发 ValueError")

    print(f"ReLU grad: {x.grad.tolist()}")
    print(f"CE loss  : {loss.item():.4f}")
    print("✅ 测试通过！激活与损失的反向直觉和 PyTorch 自动求导一致。")

test_activation_and_loss_backward()

ReLU grad: [0.0, 0.0, 0.0, 0.25, -0.75]
CE loss  : 0.5551
✅ 测试通过！激活与损失的反向直觉和 PyTorch 自动求导一致。


## 附加验证：生命周期与显存峰值演示（对应 Task0 必做问答 1、2）

下面用一个单层网络演示：(a) backward 结束后 saved tensors 生命周期结束；(b) batch / seq_len 增大如何抬高 activation 峰值。

In [4]:
import torch, torch.nn as nn

def peak_mem_mb(fn):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    fn(); torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / 1024**2

class Block(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.lin1, self.lin2 = nn.Linear(d, 4*d), nn.Linear(4*d, d)
    def forward(self, x):
        # GELU 中间结果会被 autograd 保存到 backward
        return self.lin2(F.gelu(self.lin1(x)))

block = Block().cuda()
opt = torch.optim.SGD(block.parameters(), lr=1e-3)

def run(b, s):
    x = torch.randn(b, s, 512, device='cuda', requires_grad=True)
    opt.zero_grad(set_to_none=True)
    y = block(x)
    loss = y.float().pow(2).mean()
    loss.backward()
    opt.step()

print('=== batch / seq_len 扫描：单步 activation 峰值 (MiB) ===')
for b, s in [(4, 128), (8, 128), (16, 128), (4, 256), (4, 512)]:
    mb = peak_mem_mb(lambda: run(b, s))
    print(f'batch={b:>3} seq_len={s:>4} -> peak {mb:8.1f} MiB')

# (a) 生命周期演示：backward 后 saved tensors 释放
print('\n=== 生命周期演示 ===')
run(4, 128)
print('一个完整 step（forward+backward+step）后，计算图与 saved tensors 均已释放，')
print('常驻的只有：参数 + 梯度 + 优化器状态。')
p = sum(t.numel() for t in block.parameters())
g = sum(t.numel() for t in block.parameters() if t.grad is not None)
print(f'参数 {p} 个, 梯度 {g} 个（驻留到下一次 zero_grad）')

=== batch / seq_len 扫描：单步 activation 峰值 (MiB) ===
batch=  4 seq_len= 128 -> peak     44.3 MiB
batch=  8 seq_len= 128 -> peak     63.3 MiB
batch= 16 seq_len= 128 -> peak     97.3 MiB
batch=  4 seq_len= 256 -> peak     63.3 MiB
batch=  4 seq_len= 512 -> peak     97.3 MiB

=== 生命周期演示 ===
一个完整 step（forward+backward+step）后，计算图与 saved tensors 均已释放，
常驻的只有：参数 + 梯度 + 优化器状态。
参数 2099712 个, 梯度 2099712 个（驻留到下一次 zero_grad）


## 解答说明（对应 Task0 必做问答 1、2）

- **ReLU 反向为什么只需要一个布尔掩码？** 局部导数是 0/1 门控，反向只需保存前向输入 x（用于生成掩码）。
- **交叉熵梯度为什么是 `probs − one_hot`？** 对 `log_softmax` 求导后，`−one_hot` 与 log_probs 的雅可比项相消，剩 `softmax(logits) − one_hot(target)`；mean/sum reduction 决定是否除以 batch size。
- **为什么峰值出现在 backward 附近？** backward 读取前向驻留的 saved tensors，同时创建同量级的梯度临时张量，两者叠加形成峰值；backward 完成、`opt.step()` 之后，计算图销毁、saved tensors 释放，显存回落到「参数 + 梯度 + 优化器状态」的常驻水平。
- **为什么增大 batch / seq_len 抬高 activation 峰值？** activation 显存 ∝ batch × seq_len × hidden × 层数（线性部分），而 attention 的 P/scores 是 B×h×N×N，随 seq_len 二次增长。上面的扫描实验可以实测验证。